In [4]:
import argparse
import logging
import sys
import pandas as pd

# Configure logging to write to traffic_analytics.log
logging.basicConfig(
    filename='traffic_analytics.log',
    filemode='a',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

def load_dataset(filepath='Metro_Interstate_Traffic_Volume.csv'):
    """Load and parse the traffic dataset."""
    try:
        df = pd.read_csv(filepath)
        df['date_time'] = pd.to_datetime(df['date_time'])
        return df
    except Exception as e:
        logging.error(f"Failed to load dataset from {filepath}: {e}")
        print(f"Error loading dataset: {e}")
        sys.exit(1)

def query_datetime(df, datetime_str):
    """Query 1: Retrieve traffic volume and weather for a specific timestamp."""
    try:
        target_dt = pd.to_datetime(datetime_str)
        result = df[df['date_time'] == target_dt]
        
        if result.empty:
            print(f"No traffic records found for: {datetime_str}")
        else:
            print(f"\n--- Traffic & Weather Info ({target_dt}) ---")
            for _, row in result.iterrows():
                print(f"Traffic Volume:      {row['traffic_volume']}")
                print(f"Weather Condition:   {row['weather_main']} ({row['weather_description']})")
                print(f"Temperature:         {row['temp']} K")
                print(f"Rainfall (1h):       {row['rain_1h']} mm")
                print(f"Snowfall (1h):       {row['snow_1h']} mm")
                print(f"Cloud Cover:         {row['clouds_all']}%")
                if pd.notna(row['holiday']):
                    print(f"Holiday:             {row['holiday']}")
                print("-" * 45)
    except Exception as e:
        logging.error(f"Invalid input format for query-datetime '{datetime_str}': {e}")
        print(f"Error: Invalid date/time format '{datetime_str}'. Please use format 'YYYY-MM-DD HH:MM:SS'.")

def compare_daytype(df):
    """Query 2: Compare weekday vs weekend traffic statistics."""
    df_temp = df.copy()
    df_temp['is_weekend'] = df_temp['date_time'].dt.dayofweek >= 5
    
    summary = df_temp.groupby('is_weekend')['traffic_volume'].agg(['mean', 'median', 'min', 'max'])
    
    print("\n--- Traffic Comparison: Weekday vs Weekend ---")
    print(f"Weekday Mean Volume:   {summary.loc[False, 'mean']:.2f}")
    print(f"Weekday Median Volume: {summary.loc[False, 'median']:.2f}")
    print(f"Weekend Mean Volume:   {summary.loc[True, 'mean']:.2f}")
    print(f"Weekend Median Volume: {summary.loc[True, 'median']:.2f}")

def recommend_travel(df, date_str):
    """Query 3: Identify lowest-traffic travel hours for a specific date."""
    try:
        target_date = pd.to_datetime(date_str).date()
        df_temp = df.copy()
        df_temp['date'] = df_temp['date_time'].dt.date
        day_df = df_temp[df_temp['date'] == target_date]
        
        if day_df.empty:
            print(f"No records found for date: {date_str}")
            return
            
        hourly = day_df.groupby(day_df['date_time'].dt.hour)['traffic_volume'].mean().reset_index()
        lowest_hours = hourly.sort_values(by='traffic_volume').head(3)
        
        print(f"\n--- Recommended Travel Hours for {target_date} ---")
        for _, row in lowest_hours.iterrows():
            hour = int(row['date_time'])
            print(f"Hour {hour:02d}:00 - Expected Volume: {row['traffic_volume']:.0f} vehicles")
            
    except Exception as e:
        logging.error(f"Invalid input format for recommend-travel '{date_str}': {e}")
        print(f"Error: Invalid date format '{date_str}'. Please use format 'YYYY-MM-DD'.")

def main():
    parser = argparse.ArgumentParser(description="Mini Traffic Analytics CLI Tool")
    subparsers = parser.add_subparsers(dest="command", help="Available CLI Commands")
    
    # Query 1: Query specific date/time
    p1 = subparsers.add_parser("query-datetime", help="Fetch traffic & weather for a date/time")
    p1.add_argument("--datetime", required=True, help="Date/Time string (e.g., '2012-10-02 09:00:00')")
    
    # Query 2: Compare weekdays vs weekends
    subparsers.add_parser("compare-daytype", help="Compare weekday vs weekend traffic statistics")
    
    # Query 3: Identify recommended low-traffic travel hours
    p3 = subparsers.add_parser("recommend-travel", help="Identify lowest traffic hours for a date")
    p3.add_argument("--date", required=True, help="Date string (e.g., '2012-10-02')")
    
    args = parser.parse_args(args=[] if 'ipykernel' in sys.modules else None)
    
    if args.command:
        logging.info(f"Command invoked: '{args.command}' | Arguments: {vars(args)}")
        df = load_dataset()
        
        if args.command == "query-datetime":
            query_datetime(df, args.datetime)
        elif args.command == "compare-daytype":
            compare_daytype(df)
        elif args.command == "recommend-travel":
            recommend_travel(df, args.date)
    else:
        parser.print_help()

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] {query-datetime,compare-daytype,recommend-travel} ...

Mini Traffic Analytics CLI Tool

positional arguments:
  {query-datetime,compare-daytype,recommend-travel}
                        Available CLI Commands
    query-datetime      Fetch traffic & weather for a date/time
    compare-daytype     Compare weekday vs weekend traffic statistics
    recommend-travel    Identify lowest traffic hours for a date

options:
  -h, --help            show this help message and exit


In [6]:
import sys
import os

print("Python Executable Path:", sys.executable)
print("Anaconda Root Directory Path:", sys.prefix)

Python Executable Path: C:\Users\vimih\anaconda3\python.exe
Anaconda Root Directory Path: C:\Users\vimih\anaconda3
